In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_digits
from tensorflow.keras.utils import to_categorical, plot_model
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Load and preprocess the Optdigits dataset
digits = load_digits()
images = digits.images
labels = digits.target

# Reshape and normalize the images
images = images.reshape((images.shape[0], images.shape[1], images.shape[2], 1))  
images = images.astype('float32') / 16.0  

# Convert labels to one-hot encoding
labels = to_categorical(labels, num_classes=10)

# Split the dataset into training and testing sets
train_images, test_images, train_labels, test_labels = train_test_split(images, labels, test_size=0.2, random_state=42)

In [ ]:
model = models.Sequential([
    layers.Flatten(input_shape=(8, 8, 1)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
#  Visualize the model architecture
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
print("Model architecture diagram saved as 'model_architecture.png'")

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train the model initially
model.fit(train_images, train_labels, epochs=10, batch_size=32, validation_split=0.2)

In [ ]:
# Incremental Learning
def incremental_learning(model, new_images, new_labels, epochs=1, batch_size=32):
    """
    Function to update the model with new data incrementally.
    """
    # Preprocess the new data
    new_images = new_images.reshape((new_images.shape[0], new_images.shape[1], new_images.shape[2], 1))
    new_images = new_images.astype('float32') / 16.0
    new_labels = to_categorical(new_labels, num_classes=10)

    # Train the model with the new data
    model.fit(new_images, new_labels, epochs=epochs, batch_size=batch_size, validation_split=0.2)

# Simulate the arrival of new data
new_digits = load_digits()
new_images = new_digits.images[:100]  # Take only 100 new images
new_labels = new_digits.target[:100]

# Update the model with the new data
print("Updating the model with new data...")
incremental_learning(model, new_images, new_labels, epochs=5, batch_size=32)

In [ ]:
# Evaluate the updated model on the test set
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f'Test accuracy after incremental learning: {test_acc:.4f}')


In [ ]:
# Make predictions and visualize results
predictions = model.predict(test_images)

# Show some predictions
for i in range(5):  # Show the first 5 predictions
    predicted_label = np.argmax(predictions[i])
    true_label = np.argmax(test_labels[i])
    print(f'Prediction: {predicted_label}, True Label: {true_label}')

    plt.imshow(test_images[i].reshape(8, 8), cmap='gray')
    plt.title(f'Prediction: {predicted_label}, True: {true_label}')
    plt.show()

In [ ]:
# Confusion Matrix
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(test_labels, axis=1)

# Calculate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predictions')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Dividir en 6 bloques
num_blocks = 6
block_size = len(images) // num_blocks
blocks = [(images[i*block_size:(i+1)*block_size], labels[i*block_size:(i+1)*block_size]) for i in range(num_blocks)]

history_by_block = []

# Inicializar modelo RNA
model = models.Sequential([
    layers.Flatten(input_shape=(8, 8, 1)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenamiento incremental y evaluación cruzada
for i in range(num_blocks):
    x_train, y_train = blocks[i]
    print(f"Entrenando con el bloque {i+1}")
    hist = model.fit(x_train, y_train, epochs=10, verbose=0)
    
    block_metrics = {
        'train_block': i+1,
        'train_loss': hist.history['loss'][-1],
        'train_accuracy': hist.history['accuracy'][-1],
        'eval_results': []
    }
    
    # Evaluar con todos los bloques anteriores (incluyendo el actual)
    for j in range(i+1):
        x_eval, y_eval = blocks[j]
        loss, acc = model.evaluate(x_eval, y_eval, verbose=0)
        block_metrics['eval_results'].append({
            'eval_block': j+1,
            'loss': loss,
            'accuracy': acc
        })
    history_by_block.append(block_metrics)

# Visualizar resultados
import pandas as pd
flat_results = []
for bm in history_by_block:
    for ev in bm['eval_results']:
        flat_results.append({
            'train_block': bm['train_block'],
            'eval_block': ev['eval_block'],
            'loss': ev['loss'],
            'accuracy': ev['accuracy']
        })
df = pd.DataFrame(flat_results)
print(df)
